In [1]:
import sqlite3
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# Loading experiment data
conn = sqlite3.connect('../data/hillstrom.db')
df = pd.read_sql("SELECT * FROM experiment", conn)

# Spliting into arms
treatment = df[df['treatment'] == 'Womens E-Mail']
control = df[df['treatment'] == 'No E-Mail']

# Primary metric: visit
visits_treatment = treatment['visit'].sum()
visits_control = control['visit'].sum()

n_treatment = len(treatment)
n_control = len(control)

count = [visits_treatment, visits_control]
nobs = [n_treatment, n_control]

# Two-proportion z-test
stat, pval = proportions_ztest(count, nobs)

# CI for each arm's rate
ci_treat_low, ci_treat_high = proportion_confint(visits_treatment, n_treatment, alpha=0.05)
ci_ctrl_low, ci_ctrl_high = proportion_confint(visits_control, n_control, alpha=0.05)

rate_treatment = visits_treatment / n_treatment
rate_control = visits_control / n_control
lift = rate_treatment - rate_control

# Difference in proportions CI (normal approximation)
se_diff = np.sqrt((rate_treatment * (1 - rate_treatment) / n_treatment) + 
                  (rate_control * (1 - rate_control) / n_control))
margin_error = 1.96 * se_diff
ci_lift_low = lift - margin_error
ci_lift_high = lift + margin_error

print(f"Treatment Visit Rate: {rate_treatment:.4f} (95% CI: {ci_treat_low:.4f} - {ci_treat_high:.4f})")
print(f"Control Visit Rate:   {rate_control:.4f} (95% CI: {ci_ctrl_low:.4f} - {ci_ctrl_high:.4f})")
print(f"Absolute Lift:        {lift:.4f} (95% CI: {ci_lift_low:.4f} - {ci_lift_high:.4f})")
print(f"Z-test p-value:       {pval:.4e}")

Treatment Visit Rate: 0.1514 (95% CI: 0.1466 - 0.1562)
Control Visit Rate:   0.1062 (95% CI: 0.1020 - 0.1103)
Absolute Lift:        0.0452 (95% CI: 0.0389 - 0.0516)
Z-test p-value:       3.1824e-44


The Women's E-Mail group had a 15.14% visit rate compared with 10.62% for the No E-Mail group, corresponding to an absolute lift of 4.52 percentage points (95% CI: 3.89–5.16 percentage points). The two-proportion z-test produced p < 0.001, providing strong evidence of a difference in visit rates between the two groups.

In [2]:
from scipy.stats import mannwhitneyu

# Secondary metric: conversion
conv_treatment = treatment['conversion'].sum()
conv_control = control['conversion'].sum()

stat_conv, pval_conv = proportions_ztest([conv_treatment, conv_control], nobs)
rate_conv_treat = conv_treatment / n_treatment
rate_conv_ctrl = conv_control / n_control

print("--- CONVERSION (Secondary) ---")
print(f"Treatment Conversion Rate: {rate_conv_treat:.4f}")
print(f"Control Conversion Rate:   {rate_conv_ctrl:.4f}")
print(f"Z-test p-value:            {pval_conv:.4f}\n")

# Guardrail metric: spend
spend_treat = treatment['spend'].values
spend_ctrl = control['spend'].values

# Mann-Whitney U test for rank differences
stat_mwu, pval_mwu = mannwhitneyu(spend_treat, spend_ctrl, alternative='two-sided')

# Bootstrap CI for the mean difference (5,000 resamples)
np.random.seed(42)
boot_diffs = []
for _ in range(5000):
    b_treat = np.random.choice(spend_treat, size=n_treatment, replace=True).mean()
    b_ctrl = np.random.choice(spend_ctrl, size=n_control, replace=True).mean()
    boot_diffs.append(b_treat - b_ctrl)

ci_spend_low = np.percentile(boot_diffs, 2.5)
ci_spend_high = np.percentile(boot_diffs, 97.5)
mean_spend_diff = spend_treat.mean() - spend_ctrl.mean()

print("--- SPEND (Guardrail) ---")
print(f"Treatment Mean Spend: ${spend_treat.mean():.4f}")
print(f"Control Mean Spend:   ${spend_ctrl.mean():.4f}")
print(f"Mean Difference:      ${mean_spend_diff:.4f} (95% CI: ${ci_spend_low:.4f} to ${ci_spend_high:.4f})")
print(f"Mann-Whitney p-value: {pval_mwu:.4f}")

--- CONVERSION (Secondary) ---
Treatment Conversion Rate: 0.0088
Control Conversion Rate:   0.0057
Z-test p-value:            0.0002

--- SPEND (Guardrail) ---
Treatment Mean Spend: $1.0772
Control Mean Spend:   $0.6528
Mean Difference:      $0.4244 (95% CI: $0.1799 to $0.6952)
Mann-Whitney p-value: 0.0002


In [3]:
from statsmodels.stats.proportion import proportion_confint

ci_conv_treat = proportion_confint(conv_treatment, n_treatment, alpha=0.05)
ci_conv_ctrl = proportion_confint(conv_control, n_control, alpha=0.05)
print(f"Treatment conversion CI: {ci_conv_treat}")
print(f"Control conversion CI:   {ci_conv_ctrl}")

Treatment conversion CI: (0.0075828434512433505, 0.010091444667707415)
Control conversion CI:   (0.0047129240501248785, 0.0067392490466553715)


The Women's E-Mail campaign also drove a statistically significant increase in the secondary conversion metric
(0.88% vs. 0.57%, p = 0.0002), though we note the extremely low baseline conversion rate inherent to this channel.

Although conversion occurs in under 1% of customers, the large sample size (n ≈ 21,300 per arm) still yielded enough converters in absolute terms, roughly 188 in treatment vs. 121 in control, for the z-test to detect a significant difference (p = 0.0002). The low base rate did not prevent detection here; it is a precision concern, not a power concern. With only a few hundred total events, the conversion effect size is noisier than visit's, so the point estimate should be treated as directionally reliable rather than used for exact ROI math.

For the guardrail metric of spend, the treatment group averaged $1.08 per customer compared to $0.65 for the control. The non-parametric Mann-Whitney U test (p = 0.0002) and a 5,000-iteration bootstrap confirm a statistically significant mean difference, estimating an absolute increase of $0.42 per customer (95% CI: $0.18–$0.70).

To summarise: visit, conversion, and spend all moved in the same direction and all reached statistical significance. There's no disagreement between metrics to explain here, which is itself worth stating plainly, since the original design anticipated a scenario where conversion might come back non-significant due to its rarity while visit didn't. That didn't happen though, the sample size was large enough to power both. This is also the expected causal pattern, not a coincidence; the email drives more site visits, and visits are the mechanism through which both conversions and spend increase, so all three should move together when the effect is real.